In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/mugdhanaik/phase2-step9/__notebook__.ipynb
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__output__.json
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__results__.html
/kaggle/input/notebooks/mugdhanaik/phase2-step9/custom.css
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__results___files/__results___5_0.png
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__results___files/__results___5_1.png
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__results___files/__results___5_2.png
/kaggle/input/notebooks/mugdhanaik/phase2-step9/__results___files/__results___5_3.png
/kaggle/input/notebooks/mugdhanaik/phase2-step9/p2_9/outcome_analysis.csv
/kaggle/input/notebooks/mugdhanaik/phase2-step9/p2_9/outcome_checks.csv
/kaggle/input/notebooks/mugdhanaik/phase2-step9/p2_9/outcome_counts.csv
/kaggle/input/notebooks/mugdhanaik/phase2-step9/p2_9/outcome_effects.png
/kaggle/input/notebooks/mugdhanaik/phase2-step9/p2_9/outcome_faithfulness.png
/kaggle/input/notebooks/mugdhana

In [2]:
import os, sys, json, hashlib, shutil, platform, importlib.metadata as md_
from datetime import datetime, timezone
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/Mugdha-Naik/deepfake-detector-project.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
INPUT_ROOT = "/kaggle/input"
OUT = "/kaggle/working/p2_11"
FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)
ROOTS = [INPUT_ROOT, REPO_DIR]
K_PRIMARY, CONF_MIN = 20, 0.8
NAMES = {"A": "FF++ clean", "B": "FF++ compressed (CRF 40)", "C": "Celeb-DF clean", "D": "Celeb-DF compressed (CRF 40)"}
PAIR = {"A": "A->B", "B": "A->B", "C": "C->D", "D": "C->D"}

def find_one(name, roots):
    for root in roots:
        matches = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f"{name}: {len(matches)} copies under {root}. Attach only one:\n" + "\n".join(matches))
    raise FileNotFoundError(f"{name} not found under {roots}")

USED = {}
def P(name):
    p = find_one(name, ROOTS); USED[name] = p; return p
R = lambda n: pd.read_csv(P(n))
J = lambda n: json.load(open(P(n)))
def pick(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items(): m &= (df[k] == v).to_numpy()
    if not m.any(): raise KeyError(f"no row with {kw}")
    return df[m].iloc[0]
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
print("setup OK")

Cloning into '/kaggle/working/repo'...


setup OK


In [3]:
# ============================================================
# LOAD EVERY RESULT FILE
# ============================================================
long = R("metrics_long.csv"); deltas = R("metric_deltas.csv"); flips = R("flip_summary.csv"); blur = R("blur_baseline.csv")
cam_sum = R("cam_summary.csv"); cam_chk = R("cam_checks.csv")
fa_sum = R("faithfulness_summary.csv"); del_chk = R("deletion_check.csv"); fill = R("fill_comparison.csv"); fa_chk = R("faithfulness_checks.csv")
st_sum = R("stability_summary.csv"); st_match = R("stability_floor_direction_matched.csv"); st_eff = R("stability_flip_effect.csv"); st_dist = R("stability_A_C_distribution.csv"); st_chk = R("stability_checks.csv")
out_group = R("outcome_group_summary.csv"); out_an = R("outcome_analysis.csv"); out_counts = R("outcome_counts.csv"); out_chk = R("outcome_checks.csv")
rel_sum = R("reliability_summary.csv"); rel_acc = R("reliability_accuracy.csv"); rel_chk = R("reliability_checks.csv")
metas = {"2.5 eval_meta": J("eval_meta.json"), "2.6 gradcam_meta": J("gradcam_meta.json"), "2.7 faithfulness_meta": J("faithfulness_meta.json"), "2.8 stability_meta": J("stability_meta.json"),
         "2.9 outcome_meta": J("outcome_meta.json"), "2.10 reliability_meta": J("reliability_meta.json")}
rule = J("rule_frozen.json")

# config.json is not in any attached output; only its sha256 is recorded in eval_meta
cfg = {}

# threshold.json is not in any attached output; take it from eval_meta and verify all metas agree
thr_val = metas["2.5 eval_meta"]["threshold"]
for k, m in metas.items():
    if "threshold" in m:
        assert abs(m["threshold"] - thr_val) < 1e-12, f"threshold mismatch in {k}: {m['threshold']} vs {thr_val}"
thr = {"value": thr_val, "method": metas["2.5 eval_meta"].get("threshold_method")}
print("threshold", thr_val, "consistent across", len(metas), "metas")

print("loaded", len(USED), "files")

threshold 0.22168749570846558 consistent across 6 metas
loaded 29 files


In [4]:
# ============================================================
# CONSISTENCY: all steps must come from the same model, manifest, threshold and rule
# ============================================================
cons = []
models = {k: m.get("model_sha256") for k, m in metas.items()}; mans = {k: m.get("manifest_sha256") for k, m in metas.items()}
thrs = {k: m.get("threshold") for k, m in metas.items() if m.get("threshold") is not None}
cons.append({"check": "same checkpoint hash in Steps 2.5 to 2.10", "value": len(set(models.values())), "limit": 1, "pass": len(set(models.values())) == 1 and None not in models.values()})
cons.append({"check": "same manifest hash in Steps 2.6 to 2.10", "value": len(set(v for k, v in mans.items() if v)), "limit": 1, "pass": len(set(v for v in mans.values() if v)) == 1})
cons.append({"check": "threshold is the frozen 0.2217 everywhere", "value": f"{min(thrs.values()):.6f}..{max(thrs.values()):.6f}", "limit": "0.2217 +/- 0.0005", "pass": all(abs(t - 0.2217) < 5e-4 for t in thrs.values())})
cons.append({"check": "reliability rule hash matches the frozen file", "value": rule["sha256_of_rule_text"][:12], "limit": metas["2.10 reliability_meta"]["rule_sha256"][:12], "pass": rule["sha256_of_rule_text"] == metas["2.10 reliability_meta"]["rule_sha256"]})
cons.append({"check": "rule was frozen before the Trusted share was computed", "value": f"{rule['created_utc']} <= {metas['2.10 reliability_meta']['created_utc']}", "limit": "frozen <= results", "pass": rule["created_utc"] <= metas["2.10 reliability_meta"]["created_utc"]})
for name, chk in (("2.6 cam_checks", cam_chk), ("2.7 faithfulness_checks", fa_chk), ("2.8 stability_checks", st_chk), ("2.9 outcome_checks", out_chk), ("2.10 reliability_checks", rel_chk)):
    ok = chk["pass"].astype(str).str.lower().eq("true")
    cons.append({"check": f"all rows pass in {name}", "value": f"{int(ok.sum())}/{len(chk)}", "limit": "all", "pass": bool(ok.all())})
n_long = {k: int(pick(long, cond=k, level="face", metric="acc")["n"]) for k in "ABCD"}
n_out = {k: int(pick(out_counts, cond=k, group="ALL")["n_faces"]) for k in "ABCD"}
n_rel = {k: int(pick(rel_sum, cond=k, group="ALL", variant="primary")["n_faces"]) for k in "ABCD"}
n_fa = {k: int(fa_sum[(fa_sum["cond"] == k) & (fa_sum["group"] == "ALL") & (fa_sum["fill"] == "zero") & (fa_sum["k"] == K_PRIMARY) & (fa_sum["metric"] == "faithfulness")]["n_faces"].iloc[0]) for k in "ABCD"}
cons.append({"check": "same number of faces per condition in Steps 2.5, 2.7, 2.9, 2.10", "value": str(n_long), "limit": "equal", "pass": n_long == n_out == n_rel == n_fa})
cons = pd.DataFrame(cons); print(cons.to_string(index=False))
assert cons["pass"].all(), "STEPS DO NOT BELONG TOGETHER: find which step used a different file before writing any deliverable."

                                                          check                                                  value             limit  pass
                      same checkpoint hash in Steps 2.5 to 2.10                                                      1                 1  True
                        same manifest hash in Steps 2.6 to 2.10                                                      1                 1  True
                      threshold is the frozen 0.2217 everywhere                                     0.221687..0.221687 0.2217 +/- 0.0005  True
                  reliability rule hash matches the frozen file                                           3af50ed5a3ab      3af50ed5a3ab  True
          rule was frozen before the Trusted share was computed 2026-11-07T17:35:27+00:00 <= 2026-11-07T17:35:34+00:00 frozen <= results  True
                                all rows pass in 2.6 cam_checks                                                  16/16               all  True

In [5]:
# ============================================================
# MASTER TABLE: one row per condition (accuracy ... Trusted share), all values copied from the step files
# ============================================================
def ci_cols(r, prefix, val, lo, hi):
    r[prefix], r[prefix + "_ci_lo"], r[prefix + "_ci_hi"] = val, lo, hi
rows = []
for k in "ABCD":
    r = {"cond": k, "name": NAMES[k], "twin_pair": PAIR[k]}
    r["n_faces"] = n_long[k]; r["n_videos"] = int(pick(cam_sum, cond=k, outcome="ALL")["n_videos"])
    for m in ("acc", "auc", "precision", "recall", "specificity", "f1"):
        x = pick(long, cond=k, level="face", metric=m); ci_cols(r, m, x["value"], x["ci_lo"], x["ci_hi"])
    x = pick(long, cond=k, level="video", metric="auc"); ci_cols(r, "auc_video", x["value"], x["ci_lo"], x["ci_hi"])
    for c in ("tp", "fp", "fn", "tn"): r[c] = int(pick(long, cond=k, level="face", metric=c)["value"])
    b = pick(blur, cond=k); r["blur_only_auc_face"] = b["blur_auc_face"]; r["model_beats_blur_only"] = bool(b["model_beats_blur"])
    r["twin_prediction_flip_pct"] = float(pick(flips, pair=PAIR[k])["flipped %"])
    c = pick(cam_sum, cond=k, outcome="ALL")
    for col in ("empty_share", "entropy", "central_share", "peak_share"): r["cam_" + col] = c[col]
    f = fa_sum[(fa_sum["cond"] == k) & (fa_sum["fill"] == "zero") & (fa_sum["k"] == K_PRIMARY) & (fa_sum["metric"] == "faithfulness")]
    x = f[f["group"] == "ALL"].iloc[0]; ci_cols(r, "faith_k20", x["mean"], x["ci_lo"], x["ci_hi"]); r["faith_share_positive"] = x["share_faith_positive"]
    x = f[f["group"].str.startswith("confident")].iloc[0]; ci_cols(r, "faith_k20_confident", x["mean"], x["ci_lo"], x["ci_hi"]); r["n_confident"] = int(x["n_faces"])
    xb = fa_sum[(fa_sum["cond"] == k) & (fa_sum["fill"] == "blur") & (fa_sum["k"] == K_PRIMARY) & (fa_sum["metric"] == "faithfulness") & (fa_sum["group"] == "ALL")].iloc[0]
    ci_cols(r, "faith_k20_blurfill", xb["mean"], xb["ci_lo"], xb["ci_hi"]); r["n_blurfill"] = int(xb["n_faces"])
    d_ = pick(del_chk, cond=k, k=K_PRIMARY); r["deletion_top_minus_low"] = d_["top_minus_low"]; r["deletion_share_top_hurts_more"] = d_["share_top_hurts_more"]
    r["fill_spearman_zero_vs_blur"] = pick(fill, cond=k, k=K_PRIMARY)["spearman_zero_vs_blur"]
    s = pick(st_sum, pair=PAIR[k], group="ALL"); ci_cols(r, "stab_cos", s["cosine_mean"], s["cosine_ci_lo"], s["cosine_ci_hi"])
    r["stab_floor_mean_primary"], r["stab_floor_p95_primary"] = s["floor_mean_primary"], s["floor_p95_primary"]
    r["stab_cos_class_unchanged"] = pick(st_sum, pair=PAIR[k], group="class unchanged")["cosine_mean"]
    r["stab_share_above_matched_floor_p95"] = pick(st_match, pair=PAIR[k], direction="ALL")["share_pairs_above_floor_p95"]
    r["stab_share_above_matched_floor_p95_unchanged"] = pick(st_match, pair=PAIR[k], direction="class unchanged")["share_pairs_above_floor_p95"]
    r["stab_prob_unchanged_more_similar"] = pick(st_eff, pair=PAIR[k])["prob_unchanged_more_similar"]
    for g, tag in (("ALL", "all"), ("predicted FAKE", "pred_fake"), ("predicted REAL", "pred_real")):
        x = pick(rel_sum, cond=k, group=g, variant="primary"); ci_cols(r, "trusted_share_" + tag, x["trusted_share"], x["ci_lo"], x["ci_hi"])
    r["faithful_only_share"] = pick(rel_sum, cond=k, group="ALL", variant="faith_only")["trusted_share"]; r["stable_only_share"] = pick(rel_sum, cond=k, group="ALL", variant="stab_only")["trusted_share"]
    for g, tag in (("ALL", "all"), ("predicted REAL", "pred_real"), ("predicted FAKE", "pred_fake")):
        a = rel_acc[(rel_acc["cond"] == k) & (rel_acc["group"] == g)].iloc[0]
        r[f"acc_trusted_{tag}"], r[f"acc_flagged_{tag}"] = a.get("accuracy_trusted", np.nan), a.get("accuracy_flagged", np.nan)
        r[f"acc_diff_{tag}"], r[f"acc_diff_{tag}_ci_lo"], r[f"acc_diff_{tag}_ci_hi"] = a.get("diff", np.nan), a.get("ci_lo", np.nan), a.get("ci_hi", np.nan)
        r[f"acc_diff_{tag}_reading"] = a.get("reading", "")
        r[f"n_trusted_{tag}"] = int(a["n_trusted"])
    rows.append(r)
master = pd.DataFrame(rows); master.to_csv(os.path.join(OUT, "results_master.csv"), index=False)
show = ["cond", "acc", "auc", "recall", "specificity", "f1", "faith_k20", "faith_k20_ci_lo", "faith_k20_ci_hi", "stab_cos", "stab_floor_mean_primary", "stab_share_above_matched_floor_p95", "trusted_share_all", "tp", "fp", "fn", "tn"]
print(master[show].round(3).to_string(index=False))

cond   acc   auc  recall  specificity    f1  faith_k20  faith_k20_ci_lo  faith_k20_ci_hi  stab_cos  stab_floor_mean_primary  stab_share_above_matched_floor_p95  trusted_share_all   tp   fp   fn   tn
   A 0.839 0.917   0.789        0.890 0.833      0.106            0.082            0.127     0.701                    0.673                               0.136              0.085 2368  317  632 2566
   B 0.527 0.611   0.135        0.935 0.226      0.165            0.152            0.179     0.701                    0.673                               0.136              0.104  405  186 2595 2697
   C 0.646 0.704   0.823        0.470 0.699     -0.021           -0.042            0.001     0.643                    0.617                               0.105              0.054 3605 2321  777 2060
   D 0.538 0.577   0.260        0.815 0.360      0.156            0.145            0.167     0.643                    0.617                               0.105              0.079 1139  809 3243 3572


In [6]:
# ============================================================
# BY-OUTCOME AND BY-DIRECTION TABLES
# ============================================================
rows = []
for k in "ABCD":
    for o in ("TP", "FP", "FN", "TN"):
        r = {"cond": k, "outcome": o, "pred_class": "FAKE" if o in ("TP", "FP") else "REAL"}
        c = pick(out_counts, cond=k, group=o); r["n_faces"], r["n_videos"] = int(c["n_faces"]), int(c["n_videos"])
        for met in ("faith_20", "p0", "entropy", "central_share", "stab_cos_unchanged"):
            g = pick(out_group, cond=k, group=o, metric=met); r[met], r[met + "_ci_lo"], r[met + "_ci_hi"], r[met + "_n"] = g["mean"], g["ci_lo"], g["ci_hi"], int(g["n_faces"])
        t = pick(rel_sum, cond=k, group=o, variant="primary"); r["trusted_share"], r["trusted_ci_lo"], r["trusted_ci_hi"] = t["trusted_share"], t["ci_lo"], t["ci_hi"]
        rows.append(r)
by_out = pd.DataFrame(rows); by_out.to_csv(os.path.join(OUT, "results_master_by_outcome.csv"), index=False)
by_dir = st_match[st_match["direction"].isin(["FAKE->FAKE", "FAKE->REAL", "REAL->FAKE", "REAL->REAL"])].copy()
by_dir.to_csv(os.path.join(OUT, "results_master_stability_by_direction.csv"), index=False)
print(by_out[["cond", "outcome", "n_faces", "faith_20", "p0", "stab_cos_unchanged", "trusted_share"]].round(3).to_string(index=False))

cond outcome  n_faces  faith_20    p0  stab_cos_unchanged  trusted_share
   A      TP     2368    -0.009 0.910               0.617          0.017
   A      FP      317    -0.036 0.711               0.579          0.019
   A      FN      632     0.265 0.960               0.718          0.027
   A      TN     2566     0.190 0.990               0.838          0.171
   B      TP      405    -0.035 0.591               0.616          0.037
   B      FP      186     0.006 0.577               0.584          0.032
   B      FN     2595     0.191 0.980               0.718          0.032
   B      TN     2697     0.182 0.988               0.837          0.188
   C      TP     3605    -0.188 0.889               0.606          0.007
   C      FP     2321    -0.142 0.823               0.535          0.004
   C      FN      777     0.324 0.940               0.757          0.124
   C      TN     2060     0.277 0.963               0.786          0.167
   D      TP     1139    -0.080 0.554              

In [7]:
# ============================================================
# FIGURES FOLDER (one place for the report and slides)
# ============================================================
WANTED = ["cleaning_dropoff_chart.png", "pair_samples.png", "training_curves.png", "roc_curves.png", "confusion_matrices.png", "score_distributions.png",
          "mean_heatmaps.png", "gradcam_examples_A.png", "gradcam_examples_B.png", "gradcam_examples_C.png", "gradcam_examples_D.png", "gradcam_twins_A_B.png", "gradcam_twins_C_D.png",
          "faithfulness_by_condition.png", "faithfulness_hist.png", "faithfulness_by_outcome.png", "stability_hist_pairs.png", "stability_by_group.png", "mean_maps_by_target.png",
          "outcome_faithfulness.png", "outcome_stability.png", "outcome_stability_unchanged.png", "outcome_effects.png", "outcome_panels_A.png", "outcome_panels_C.png",
          "trusted_share.png", "trusted_by_outcome.png", "trusted_accuracy.png"]
idx = []
for name in WANTED:
    try:
        p = find_one(name, ROOTS); shutil.copy(p, os.path.join(FIG, name)); idx.append({"figure": name, "found": True, "source": p})
    except (FileNotFoundError, RuntimeError) as e:
        idx.append({"figure": name, "found": False, "source": str(e)[:80]})
fig_index = pd.DataFrame(idx); fig_index.to_csv(os.path.join(OUT, "figures_index.csv"), index=False)
print(f"{int(fig_index['found'].sum())}/{len(fig_index)} figures copied. Missing:", list(fig_index.loc[~fig_index['found'], 'figure']))

23/28 figures copied. Missing: ['cleaning_dropoff_chart.png', 'pair_samples.png', 'training_curves.png', 'outcome_panels_A.png', 'outcome_panels_C.png']


In [8]:
# ============================================================
# NUMBERS SHEET: every headline number with its source file (copy numbers from here, never from memory)
# ============================================================
def f3(v, lo=None, hi=None, d=3):
    s = f"{v:.{d}f}"
    return s if lo is None or pd.isna(lo) else f"{s} [{lo:.{d}f}, {hi:.{d}f}]"
L = ["# Numbers sheet (generated by Step 2.11; do not edit by hand)", f"Generated {datetime.now(timezone.utc).isoformat(timespec='seconds')} UTC from the saved result files. Threshold {thr['value']:.4f} (frozen). Manifest {metas['2.6 gradcam_meta']['manifest_sha256'][:8]}. Checkpoint {metas['2.6 gradcam_meta']['model_sha256'][:12]}.", ""]
try:
    fcounts = pd.read_csv(find_one("final_counts.csv", ROOTS)); L += ["## Phase 1 final counts (final_counts.csv)", fcounts.to_markdown(index=False) if hasattr(fcounts, "to_markdown") else fcounts.to_string(index=False), ""]
except Exception:
    L += ["## Phase 1 final counts", "(final_counts.csv not attached: copy the Phase 1 numbers from the repo `data/phase1/cleaning/`)", ""]
L += ["## Detector (Step 2.5; face level, frozen threshold, 95% CI by resampling videos) [source: metrics_long.csv]", "| Cond | Accuracy | ROC-AUC | Recall | Specificity | F1 | Video-level AUC | Blur-only AUC | TP / FP / FN / TN |", "|---|---|---|---|---|---|---|---|---|"]
for _, r in master.iterrows():
    L.append(f"| {r.cond} {r['name']} | {f3(r.acc, r.acc_ci_lo, r.acc_ci_hi)} | {f3(r.auc, r.auc_ci_lo, r.auc_ci_hi)} | {f3(r.recall, r.recall_ci_lo, r.recall_ci_hi)} | {f3(r.specificity, r.specificity_ci_lo, r.specificity_ci_hi)} | {f3(r.f1, r.f1_ci_lo, r.f1_ci_hi)} | {f3(r.auc_video, r.auc_video_ci_lo, r.auc_video_ci_hi)} | {r.blur_only_auc_face:.3f} (model beats it: {r.model_beats_blur_only}) | {r.tp} / {r.fp} / {r.fn} / {r.tn} |")
L += ["", "Twin prediction flips under compression [flip_summary.csv]: " + "; ".join(f"{p} {float(pick(flips, pair=p)['flipped %']):.2f}% (REAL->FAKE {int(pick(flips, pair=p)['REAL->FAKE flips'])}, FAKE->REAL {int(pick(flips, pair=p)['FAKE->REAL flips'])})" for p in ("A->B", "C->D")), "",
      "## Grad-CAM (Step 2.6) [cam_summary.csv, cam_checks.csv]", "Checks passed: " + f"{int(cam_chk['pass'].astype(str).str.lower().eq('true').sum())}/{len(cam_chk)}. See cam_checks.csv for the CAM equivalence error and the randomisation correlation.", "",
      "## Faithfulness (Step 2.7; k=20%, zero fill; 95% CI over videos) [faithfulness_summary.csv]", "| Cond | Score (all faces) | Score (confident p0>=0.8, n) | Blur fill (n faces) | Share of faces with score >0 | Deletion check: top minus lowest | Spearman zero vs blur |", "|---|---|---|---|---|---|---|"]
for _, r in master.iterrows():
    L.append(f"| {r.cond} | {f3(r.faith_k20, r.faith_k20_ci_lo, r.faith_k20_ci_hi)} (accuracy {r.acc:.3f}) | {f3(r.faith_k20_confident, r.faith_k20_confident_ci_lo, r.faith_k20_confident_ci_hi)} ({r.n_confident}) | {f3(r.faith_k20_blurfill, r.faith_k20_blurfill_ci_lo, r.faith_k20_blurfill_ci_hi)} ({r.n_blurfill}) | {r.faith_share_positive:.3f} | {r.deletion_top_minus_low:.3f} (top hurts more in {r.deletion_share_top_hurts_more * 100:.0f}% of faces) | {r.fill_spearman_zero_vs_blur:.2f} |")
L += ["", "## Stability (Step 2.8) [stability_summary.csv, stability_floor_direction_matched.csv]", "A and B share the A->B numbers; C and D share the C->D numbers (same pairs).", "| Pair | Mean cosine (95% CI) | Floor mean / p95 (first floor) | Class-unchanged cosine | Share above matched floor p95 (all / class unchanged) | P(unchanged more similar) |", "|---|---|---|---|---|---|"]
for p in ("A->B", "C->D"):
    r = master[master["twin_pair"] == p].iloc[0]
    L.append(f"| {p} | {f3(r.stab_cos, r.stab_cos_ci_lo, r.stab_cos_ci_hi)} | {r.stab_floor_mean_primary:.3f} / {r.stab_floor_p95_primary:.3f} | {r.stab_cos_class_unchanged:.3f} | {r.stab_share_above_matched_floor_p95:.3f} / {r.stab_share_above_matched_floor_p95_unchanged:.3f} | {r.stab_prob_unchanged_more_similar:.2f} |")
L += ["", "Direction-matched floor (same explained-class combination, different video):", "| Pair | Direction | n pairs | Paired mean | Floor mean | Floor p95 | Share above p95 |", "|---|---|---|---|---|---|---|"]
for _, r in by_dir.iterrows():
    L.append(f"| {r.pair} | {r.direction} | {int(r.n_pairs)} | {f3(r.paired_mean, r.paired_ci_lo, r.paired_ci_hi)} | {r.floor_mean:.3f} | {r.floor_p95:.3f} | {r.share_pairs_above_floor_p95:.3f} |")
L += ["", "## Outcome groups (Step 2.9) [results_master_by_outcome.csv, outcome_analysis.csv]", "| Cond | Outcome | n faces | Faithfulness | Confidence p0 | Stability (class unchanged) | Trusted share |", "|---|---|---|---|---|---|---|"]
for _, r in by_out.iterrows():
    L.append(f"| {r.cond} | {r.outcome} | {r.n_faces} | {f3(r.faith_20, r.faith_20_ci_lo, r.faith_20_ci_hi)} | {r.p0:.3f} | {r.stab_cos_unchanged:.3f} (n {r.stab_cos_unchanged_n}) | {f3(r.trusted_share, r.trusted_ci_lo, r.trusted_ci_hi)} |")
n_diff = int(out_an["reading"].astype(str).str.startswith("differs").sum()); n_tested = int(out_an["n_comparisons_tested"].iloc[0])
L += ["", f"Comparisons tested (Holm family): {n_tested}; flagged as differing: {n_diff}. Read the effect sizes in outcome_analysis.csv before quoting any of them.", "",
      "## Reliability rule (Step 2.10) [reliability_summary.csv, reliability_accuracy.csv, rule_frozen.json]",
      f"Rule frozen at {rule['created_utc']} UTC (hash {rule['sha256_of_rule_text'][:12]}). Primary: {rule['primary']['faithful']} AND {rule['primary']['stable']}.", f"Disclosure: {rule['disclosure']}", "",
      "| Cond | Trusted share (all) | Predicted FAKE | Predicted REAL | Faithful only | Stable only | Accuracy Trusted vs Flagged (predicted REAL) | n trusted (predicted FAKE) |", "|---|---|---|---|---|---|---|---|"]
for _, r in master.iterrows():
    L.append(f"| {r.cond} | {f3(r.trusted_share_all, r.trusted_share_all_ci_lo, r.trusted_share_all_ci_hi)} | {f3(r.trusted_share_pred_fake, r.trusted_share_pred_fake_ci_lo, r.trusted_share_pred_fake_ci_hi)} | {f3(r.trusted_share_pred_real, r.trusted_share_pred_real_ci_lo, r.trusted_share_pred_real_ci_hi)} | {r.faithful_only_share:.3f} | {r.stable_only_share:.3f} | {r.acc_trusted_pred_real:.3f} vs {r.acc_flagged_pred_real:.3f} (diff {f3(r.acc_diff_pred_real, r.acc_diff_pred_real_ci_lo, r.acc_diff_pred_real_ci_hi)}; {r.acc_diff_pred_real_reading}) | {r.n_trusted_pred_fake} |")
L += ["", "## Limitations to state in the report (with the numbers that justify them)",
      "- One architecture (XceptionNet); Grad-CAM here equals classic CAM for this head, and the maps are 10x10, so they are coarse.",
      f"- Masking artefacts: faithfulness depends on the fill (Spearman between zero fill and blur fill: {master['fill_spearman_zero_vs_blur'].min():.2f} to {master['fill_spearman_zero_vs_blur'].max():.2f}). Scores follow the predicted class (see by-outcome table), so read them next to the class mix.",
      "- B and D are a double-compression approximation (CRF 40 re-encoding of a C23 mirror). Detector AUC there: " + ", ".join(f"{c} {master.loc[master['cond'] == c, 'auc'].iloc[0]:.3f}" for c in "BD") + "; beats the blur-only baseline: " + ", ".join(f"{c} {bool(master.loc[master['cond'] == c, 'model_beats_blur_only'].iloc[0])}" for c in "ABCD") + ".",
      f"- The FF++ threshold over-flags real Celeb-DF faces (specificity {master.loc[master['cond'] == 'C', 'specificity'].iloc[0]:.2f} in C).",
      f"- Test AUC on A is {master.loc[master['cond'] == 'A', 'auc'].iloc[0]:.3f}; compare with the validation AUC in training_log.csv (validation chose the epoch and the threshold, so the test value is the honest one).",
      f"- The stability criterion needs a clean/compressed twin of the same face, so the Trusted rule is an evaluation tool, not a rule for single images. Twin pairs whose prediction flipped under compression: {master.loc[master['cond'] == 'A', 'twin_prediction_flip_pct'].iloc[0]:.1f}% (A->B) and {master.loc[master['cond'] == 'C', 'twin_prediction_flip_pct'].iloc[0]:.1f}% (C->D); such pairs are almost always Flagged.",
      "- The B and D rows of stability and of the stability part of the outcome analysis repeat the same pairs as A and C; they are not independent evidence.",
      "- A vs C stability is at distribution level only (no shared faces). Small groups are marked descriptive in the CSVs.",
      "- The reliability rule has the form suggested in the plan; aggregate results of Steps 2.7 to 2.9 had been seen when it was written (disclosure above). Its Trusted share was computed only after it was frozen.",
      "- Step 1 to 3 integrity checks are simple (file opens, exact MD5 duplicates, folder names), and the SHAP check was not run."]
with open(os.path.join(OUT, "numbers_sheet.md"), "w") as f: f.write("\n".join(L))
print("numbers_sheet.md written:", len(L), "lines")

numbers_sheet.md written: 90 lines


In [9]:
# ============================================================
# REPRODUCIBILITY RECORD
# ============================================================
def ver(pkg):
    try: return md_.version(pkg)
    except Exception: return None
git_head = os.popen(f"cd {REPO_DIR} && git rev-parse HEAD 2>/dev/null").read().strip() or None
repro = {"created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "python": platform.python_version(),
         "packages": {p: ver(p) for p in ("torch", "torchvision", "timm", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pillow", "facenet-pytorch", "opencv-python")},
         "repo_commit_at_consolidation": git_head, "manifest_sha256": metas["2.6 gradcam_meta"]["manifest_sha256"], "checkpoint_sha256": metas["2.6 gradcam_meta"]["model_sha256"],
         "threshold": float(thr["value"]), "reliability_rule_sha256": rule["sha256_of_rule_text"], "reliability_rule_frozen_at": rule["created_utc"],
         "training": {k: cfg.get(k) for k in ("best_epoch", "train_faces", "seed", "mean", "std", "lr_stage1", "lr_stage2", "batch_size") if k in cfg},
         "seeds": {"faithfulness": metas["2.7 faithfulness_meta"].get("seed"), "stability": metas["2.8 stability_meta"].get("seed"), "outcomes": metas["2.9 outcome_meta"].get("seed"), "reliability": metas["2.10 reliability_meta"].get("seed")},
         "faithfulness": {k: metas["2.7 faithfulness_meta"].get(k) for k in ("ks", "k_primary", "R", "fill_primary", "fill_second", "target_class", "probability", "mask_resolution")},
         "notebook_run_order": ["step6 (updated)", "step5 (scaled)", "step7", "step8", "2.1 manifest", "2.2 compressed B/D (phase2_step2)", "2.3/2.4 model and training", "2.5 p2_5_predictions_metrics",
                                "2.6 p2_6_gradcam", "2.7 p2_7_faithfulness", "2.8 p2_8_stability (re-run with matched floor)", "2.9 p2_9_outcome_analysis", "2.10 p2_10_reliability", "2.11 p2_11_consolidate"],
         "frozen_decisions": ["threshold 0.2217 chosen once on validation", "REAL=0 FAKE=1", "no retraining after test results", "manifest never regenerated", "reliability rule frozen before Trusted share"],
         "input_file_sha256": {n: sha256(p) for n, p in sorted(USED.items())}}
json.dump(repro, open(os.path.join(OUT, "reproducibility.json"), "w"), indent=2)
print({k: v for k, v in repro.items() if k not in ("input_file_sha256",)})

{'created_utc': '2026-11-09T15:00:35+00:00', 'python': '3.13.16', 'packages': {'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'timm': '1.0.30', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0', 'pillow': '13.0.0', 'facenet-pytorch': None, 'opencv-python': '4.14.0.94'}, 'repo_commit_at_consolidation': '3522e7d91a8cf45aa688a99ecc443036cf878166', 'manifest_sha256': '80eae250d797355a0e27180c7afa5753b642ce82554bf58bbf9a8391f95218f6', 'checkpoint_sha256': '026889a837586b4824c64bd61f32dbddb790a2cb83483b0fd6130d79c9d7c85c', 'threshold': 0.22168749570846558, 'reliability_rule_sha256': '3af50ed5a3abe2fdab22667343d469e69895d7ae44b58b083993dc3e97baf10e', 'reliability_rule_frozen_at': '2026-11-07T17:35:27+00:00', 'training': {}, 'seeds': {'faithfulness': 2026, 'stability': 2026, 'outcomes': 2026, 'reliability': 2026}, 'faithfulness': {'ks': [10, 20, 30], 'k_primary': 20, 'R': 5, 'fill_primary': 'zero (mid-grey in normalised space)', 'f

In [10]:
# ============================================================
# FINAL CHECKS
# ============================================================
checks = [{"check": "results_master has 4 conditions", "value": len(master), "limit": 4, "pass": len(master) == 4}]
key = ["acc", "auc", "recall", "specificity", "f1", "faith_k20", "stab_cos", "trusted_share_all", "tp", "fp", "fn", "tn"]
checks.append({"check": "no missing key values in results_master", "value": int(master[key].isna().sum().sum()), "limit": 0, "pass": bool(master[key].notna().all().all())})
checks.append({"check": "TP+FP+FN+TN equals n_faces", "value": "all rows", "limit": "equal", "pass": bool(((master.tp + master.fp + master.fn + master.tn) == master.n_faces).all())})
checks.append({"check": "numbers sheet and reproducibility record written", "value": "yes", "limit": "yes", "pass": os.path.exists(os.path.join(OUT, "numbers_sheet.md")) and os.path.exists(os.path.join(OUT, "reproducibility.json"))})
checks.append({"check": "figures copied", "value": f"{int(fig_index['found'].sum())}/{len(fig_index)}", "limit": "informational (missing ones are listed in figures_index.csv)", "pass": True})
checks = pd.concat([cons, pd.DataFrame(checks)], ignore_index=True); checks.to_csv(os.path.join(OUT, "consolidation_checks.csv"), index=False)
print(checks.to_string(index=False)); print("\nsaved:", sorted(os.listdir(OUT)))
assert checks["pass"].all(), "A CONSOLIDATION CHECK FAILED."
print("\nDone. Next: tag the repo phase2-v1, update README and the deck, then Phase 3 (report, slides, defense). Do NOT change any result.")

                                                          check                                                  value                                                        limit  pass
                      same checkpoint hash in Steps 2.5 to 2.10                                                      1                                                            1  True
                        same manifest hash in Steps 2.6 to 2.10                                                      1                                                            1  True
                      threshold is the frozen 0.2217 everywhere                                     0.221687..0.221687                                            0.2217 +/- 0.0005  True
                  reliability rule hash matches the frozen file                                           3af50ed5a3ab                                                 3af50ed5a3ab  True
          rule was frozen before the Trusted share was computed 2026-1